In [ ]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np

#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

2025-08-20 11:17:31.302779: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-20 11:17:32.266221: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-08-20 11:17:32.266249: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [1]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='8GB')
client=Client(cluster)

In [2]:
path="/gpfs/gibbs/pi/reilly/tabula_data/shendure"
name="ortho_primordial_v3"
data_root="/gpfs/gibbs/pi/reilly/tabula_data"
primordial=scm.ortho.load(client,path,name)

In [4]:
scm.SHENDURE_BOUNDS#.cells_per_cell_type

Bounds(metadata=None, min_mpra_umi=0.0034641979561541277, max_mpra_umi=369.88473782763344, by_cre_theta=0.62110305, by_cell_type_theta=0.35434753, by_cre_zi=0.39563528657381514, by_cell_type_zi=0.022971882287718185, num_cres=208, cells_per_cell_type=cell_type
Cardiomyocytes              680
EpiblastPrimitiveStreak    3445
ExEndodermParietal         4644
ExEndodermVisceral         3238
Haematoendothelial         1079
Mesoderm                   7427
NeuroectodermBrain         7750
NeuroectodermRostral       1757
SurfaceEctoderm            5168
reference                  8201
Name: cells_per_cell_type, dtype: int64, transfection_nb_mu=18.010555670792133, transfection_nb_alpha=0.5705626300443596)

Here's an example of creating an artificial experiment.

In [ ]:
#first, we define the new parameters we want to assign to this object.
new_cell_number=pd.Series({"reference":1000,"blood":2000,"neuron":1000})
new_min=0
new_max=200
new_zi=0.05

new_MOI=30

In [ ]:
artificial_bounds=scm.SHENDURE_BOUNDS.copy(
    min_mpra_umi=new_min,
    max_mpra_umi=new_max,
    zi=new_zi)
artificial_bounds.set_effective_moi(new_MOI)

18.010555670792133

In [8]:
artificial_bounds.set_effective_moi(100)

In [9]:
spread=scm.simple_spread(new_cell_number.keys(),1,450)

In [ ]:
def description_from_bounds(experiment_bounds:scm.Bounds,
                            spread:pd.DataFrame):
    """
    Returns a primordial description dask dataframe from bounds
    and ground truth dataframe. 

    See README spec for details on ground truth dataframe.
    You can easially create one with the helper function `simple_spread`. 
    """

    #known before you start or "to be optimized":
    #  cells per cell-type is a fixed parameter
    #  barcodes per CRE is a fixed parameter
    
    #for each cell type, cell, decide how many MPRA barcodes are transfected.
    #note that MOI here is measured, not applied MOI. Transfection inefficiencies
    #Will effectively reduce MOI from what is applied. 
    #probably the way to do it is
    # - calculate fraction of cells with 0, 1, 2, 3... barcodes
    # - multiple fractions by total number of cells & round
    # - make a table with n=total_cells cell barcodes
    # - add number of barcodes in each cell as a column according to fraction (n_transfected).
    # - randomize row order
    # - assign cell_type based on proportions
    # - randomly sample n_transfected MPRA barcodes for each row, then convert to tall
    # - merge in CRE identity 

    #for a non-tfection reporter setup, extend to 'all-by-all' (all mpra BC by all) filling in zeroes

    #The output will look like an ortho describe_primordial description dataframe. 
    pass


In [ ]:
description_from_bounds(experiment_bounds=)

In [29]:
cluster.close()